# Tạo Natural Queries cho Retrieval Evaluation

## Mục tiêu

Tạo query tự nhiên như người dùng thực sự tìm kiếm trên search bar.
**Không** sử dụng product_id, title nguyên văn hay bất kỳ metadata nào của sản phẩm.

## Các loại query tự nhiên

1. **Mô tả ngắn** - người dùng gõ từ khóa chính
2. **Câu hỏi** - tìm kiếm dạng hỏi đáp
3. **Tên viết tắt** - brand + product type
4. **Tình huống sử dụng** - use case cụ thể
5. **So sánh** - tìm kiếm để so sánh

In [ ]:
import json
import random
from pathlib import Path
import pandas as pd
import ast

# Cấu hình
DATA_DIR = Path("/content/llm_provider_benchmarking/embedding_project/data")
OUTPUT_DIR = Path("/content/llm_provider_benchmarking/embedding_project/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

# Template query tự nhiên - như người dùng search thật sự
QUERY_TEMPLATES = {
    # Ngắn gọn - keyword search
    "keyword": [
        "{product_type}",
        "{brand} {product_type}",
        "mua {product_type}",
        "{product_type} giá rẻ",
        "{product_type} tốt nhất",
    ],
    
    # Câu hỏi tự nhiên
    "question": [
        "{product_type} nào tốt",
        "tìm {product_type} cho {use_case}",
        "{product_type} cho {use_case}",
        "nên mua {product_type} nào",
        "gợi ý {product_type}",
    ],
    
    # Brand + type
    "brand_type": [
        "{brand} {product_type}",
        "{brand} {color} {product_type}",
        "{brand} size {size} {product_type}",
    ],
    
    # Use case
    "use_case": [
        "{product_type} cho {use_case}",
        "{product_type} dành cho {use_case}",
        "{size} {product_type} {use_case}",
        "{product_type} {use_case}",
    ],
    
    # Mua sắm
    "shopping": [
        "mua {product_type} online",
        "đặt mua {product_type}",
        "shop bán {product_type}",
        "bán {product_type} chính hãng",
    ],
}

USE_CASES = [
    "nam", "nữ", "bé trai", "bé gái",
    "người lớn tuổi", "trẻ em",
    "du lịch", "đi làm", "ở nhà",
    "tập gym", "chạy bộ", "bơi lội",
    "nấu ăn", "làm việc", "ngủ",
]

COLORS = [
    "đen", "trắng", "đỏ", "xanh", "hồng",
    "vàng", "cam", "tím", "nâu", "xám",
    "xanh navy", "be", "kem", "bạc", "vàng gold",
]

SIZES = ["nhỏ", "vừa", "lớn", "XL", "XXL", "size S", "size M", "size L"]

In [ ]:
def extract_product_type(title, category_leaf):
    """Trích xuất product type từ title/category."""
    # Lấy từ category leaf nếu có
    if category_leaf and category_leaf != "nan":
        return category_leaf.lower()
    
    # Fallback: lấy từ title
    words = str(title).split()
    if len(words) > 3:
        return " ".join(words[:3]).lower()
    return str(title).lower()


def generate_natural_query(row):
    """Tạo query tự nhiên như người dùng search thật."""
    brand = str(row.get("brand", "")).strip() if pd.notna(row.get("brand")) else ""
    if brand == "nan" or brand == "No Brand" or brand == "No brands":
        brand = ""
    
    category = str(row.get("category_leaf", "")).strip() if pd.notna(row.get("category_leaf")) else ""
    title = str(row.get("title", ""))
    
    product_type = extract_product_type(title, category)
    
    # Chọn loại query ngẫu nhiên
    query_type = random.choice(list(QUERY_TEMPLATES.keys()))
    templates = QUERY_TEMPLATES[query_type]
    template = random.choice(templates)
    
    # Fill template
    use_case = random.choice(USE_CASES)
    color = random.choice(COLORS)
    size = random.choice(SIZES)
    
    # Brand ngẫu nhiên - có hoặc không
    use_brand = brand and random.random() > 0.3
    query_brand = brand if use_brand else ""
    
    try:
        query = template.format(
            product_type=product_type,
            brand=query_brand,
            use_case=use_case,
            color=color,
            size=size,
        )
    except:
        query = product_type
    
    # Clean
    query = " ".join(query.split())
    query = query.replace("  ", " ")
    
    return {
        "query": query,
        "query_type": query_type,
        "product_id": str(row["product_id"]),
        "title": title,
        "brand": brand,
        "category": category,
    }

In [ ]:
# Tải corpus
corpus_path = DATA_DIR / "ecommerce.csv"
print(f"Loading corpus from {corpus_path}...")

df = pd.read_csv(corpus_path)
df = df.dropna(subset=["product_id", "searchable_text"])
df = df.drop_duplicates(subset=["product_id"])

# Parse category
def parse_category(cat_str):
    if pd.isna(cat_str):
        return []
    try:
        return ast.literal_eval(cat_str)
    except:
        return []

df["category_list"] = df["category"].apply(parse_category)
df["category_leaf"] = df["category_list"].apply(lambda x: x[-1] if x else "")

print(f"Loaded {len(df)} products")

In [ ]:
# Generate queries
NUM_QUERIES = 500
print(f"Generating {NUM_QUERIES} natural queries...")

# Chọn ngẫu nhiên sản phẩm
sampled_indices = random.sample(range(len(df)), min(NUM_QUERIES, len(df)))
sampled_df = df.iloc[sampled_indices].reset_index(drop=True)

queries = []
for idx, row in sampled_df.iterrows():
    q = generate_natural_query(row)
    queries.append(q)

print(f"Generated {len(queries)} queries")

In [ ]:
# Hiển thị sample
print("\n" + "="*80)
print("SAMPLE NATURAL QUERIES")
print("="*80 + "\n")

for i in range(min(20, len(queries))):
    q = queries[i]
    print(f"[{q['query_type']:10}] \"{q['query']}\"")
    print(f"           -> {q['title'][:60]}...")
    print()

In [ ]:
# Thống kê query type
from collections import Counter

type_counts = Counter(q["query_type"] for q in queries)

print("\n" + "="*40)
print("QUERY TYPE DISTRIBUTION")
print("="*40)
for qt, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    pct = count / len(queries) * 100
    print(f"  {qt:15}: {count:4} ({pct:5.1f}%)")

In [ ]:
# Lưu queries
output_file = OUTPUT_DIR / "natural_queries_500.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for q in queries:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print(f"\nSaved to: {output_file}")
print(f"Total queries: {len(queries)}")

## Bước tiếp theo

Upload file `natural_queries_500.jsonl` lên Google Drive, sau đó chạy notebook **baseline_comparison_natural_queries.ipynb** trên Colab để:

1. So sánh BM25 vs E5 pretrained vs E5 fine-tuned
2. Đánh giá semantic retrieval thực sự
3. Xem kết quả có khác gì so với query dạng literal title